# Fine-tune NandiBERT on GLUE: MRPC paraphrase detection

MRPC contains pairs of sentences. The task is to decide whether they express the same meaning.

We start from the released pretrained checkpoint. We are **not** pretraining a BERT model. The task-specific classification output is trained using labelled examples, and the encoder is fine-tuned as well.


## 1. Install and import

Run this once in a fresh environment. The NandiBERT model card specifies Transformers 5.x and `trust_remote_code=True`.

In [ ]:
!pip install -U "transformers>=5.0" "datasets<4" accelerate scikit-learn

In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, Trainer, TrainingArguments
)
from sklearn.metrics import accuracy_score, f1_score

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 2. Load the data

First inspect a row. Sentence classification gives one label to a whole sentence; sentence-pair tasks give one label to a pair.

In [ ]:
dataset = load_dataset("nyu-mll/glue", "mrpc")
print(dataset)
print(dataset["train"][0])

## 3. Tokenize

The tokenizer converts text into token IDs and builds an attention mask. Padding is done later by the data collator, so each batch is padded only to its longest example.

In [ ]:
MODEL_ID = "FrontiersMind/NandiBERT-150M"
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, trust_remote_code=True
)

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["sentence1"],
        batch["sentence2"],
        truncation=True,
        max_length=256
    )

tokenized = dataset.map(tokenize, batched=True)

## 4. Load the classification model


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    trust_remote_code=True,
    dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float32,
)
print("Parameters:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

## 5. Metrics and training

Accuracy is easy to interpret. F1 is useful when classes are not evenly represented. For MRPC we report both accuracy and F1; for MNLI, accuracy is the standard GLUE metric.

In [ ]:
def compute_metrics(output):
    predictions = output.predictions.argmax(axis=-1)
    return {
        "accuracy": accuracy_score(output.label_ids, predictions),
        "f1": f1_score(output.label_ids, predictions),
    }

In [ ]:
args = TrainingArguments(
    output_dir="./results/mrpc",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

## 6. Evaluate and save

For MNLI we evaluate both matched and mismatched validation sets. For the other tasks, we evaluate the validation split.

In [ ]:
print(trainer.evaluate(tokenized["validation"]))

In [ ]:
trainer.save_model("./results/mrpc/final")
tokenizer.save_pretrained("./results/mrpc/final")

## What to try next

- Compare learning rates such as `1e-5`, `2e-5` and `3e-5`.
  
I'm excited to see what the community will do with this model and I'm looking forward to seeing what you all build with it!

Until next time, happy coding!